In [8]:
import sqlite3
import re

DB_PATH = "db/bible.db"

def segment_english(text: str) -> list:
    """
    英文分词（只保留纯单词，统一小写）
    """
    if not text:
        return []
    # 去掉标点，只保留字母和数字
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return [w.lower() for w in text.split() if w]


def build_words_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # ✅ 创建 words 表（加约束）
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS words (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        verse_id TEXT NOT NULL,
        word TEXT NOT NULL,
        order_index INTEGER NOT NULL,
        UNIQUE(verse_id, order_index)
    )
    """)

    # ✅ 获取所有英文经文
    cursor.execute("""
        SELECT id, text_en
        FROM verse
        WHERE text_en IS NOT NULL
    """)
    verses = cursor.fetchall()

    inserted = 0
    skipped = 0

    for verse_id, text_en in verses:
        # ✅ 如果该 verse 已分词，跳过
        cursor.execute(
            "SELECT 1 FROM words WHERE verse_id = ? LIMIT 1",
            (verse_id,)
        )
        if cursor.fetchone():
            skipped += 1
            continue

        words = segment_english(text_en)

        for idx, word in enumerate(words):
            cursor.execute("""
                INSERT INTO words (verse_id, word, order_index)
                VALUES (?, ?, ?)
            """, (verse_id, word, idx))
            inserted += 1

    conn.commit()
    conn.close()

    print("✅ words 表英文分词完成")
    print(f"   新增单词：{inserted}")
    print(f"   跳过章节：{skipped}")


# ✅ 一键运行
build_words_table()

✅ words 表英文分词完成
   新增单词：531
   跳过章节：0


In [7]:
# 清空数据

import sqlite3

DB_PATH = "db/bible.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DELETE FROM words;")
conn.commit()
conn.close()

print("✅ words 表数据已全部清空")

✅ words 表数据已全部清空
